# ThermoFuzzy-PID · 01 · Thermal model & system identification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fobos123deimos/thermofuzzy-pid/blob/main/notebooks/01_thermal_model_identification.ipynb)

From the first-order-plus-dead-time model of the burn-in chamber to a reproducible synthetic step-test and nonlinear least-squares identification.

> **Data provenance:** experiments use deterministic synthetic thermal signals and parameters reported in a 2021 undergraduate thesis. No original sensor time series have been recovered; numerical results do not reproduce or validate the published curves.

## Read the model and method visually

[Six-page vector atlas](https://fobos123deimos.github.io/thermofuzzy-pid/assets/thermal-control-atlas.pdf) · [Gallery and editable TikZ](https://github.com/fobos123deimos/thermofuzzy-pid/blob/main/docs/DIAGRAMS.md)

These original explanatory diagrams distinguish source-reported parameters, reconstruction assumptions and synthetic evidence. Cabinet geometry and sensor positions are illustrative.

![Burn-in chamber: physical picture and reduced model](https://raw.githubusercontent.com/fobos123deimos/thermofuzzy-pid/main/docs/figures/tikz/01-chamber-model.png)

Illustrative cabinet, rack, heater and four named sensor channels, alongside the one-state model and allocated measurement delay. Geometry and sensor positions are not recovered hardware dimensions.

![Identification: excite, estimate, then challenge the model](https://raw.githubusercontent.com/fobos123deimos/thermofuzzy-pid/main/docs/figures/tikz/02-identification.png)

The four source-reported FOPDT parameter sets, synthetic in-sample fitting and a recommended independent validation path. The integer-delay ARX mapping is distinguished from the FOPDT step-fit experiment; notebooks 01/05 do not execute the recommended split.


In [ ]:
# Import local editable source, or install the repository in Google Colab.
from pathlib import Path
import sys, subprocess
candidates = [Path.cwd() / "src", Path.cwd().parent / "src"]
local_src = next((p for p in candidates if (p / "burnin_pid_fuzzy").exists()), None)
if local_src is not None:
    sys.path.insert(0, str(local_src.resolve()))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "git+https://github.com/fobos123deimos/thermofuzzy-pid.git"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from burnin_pid_fuzzy.models import (Plant, Scenario, DoorEvent, PLANTS_FROM_THESIS,
                                      fopdt_step_response, simulate, simulate_methods)
from burnin_pid_fuzzy.controllers import FuzzyPID, FixedPID, imc_pid_gains
from burnin_pid_fuzzy.fuzzy import FuzzyRuleBase, infer_normalized_gains, membership_degrees
from burnin_pid_fuzzy.metrics import evaluate
np.set_printoptions(precision=4, suppress=True)


## 1. Physical model and thesis provenance

The 2021 undergraduate thesis used a first-order-plus-dead-time approximation for the **combined plant and sensor** response (Eq. 2.1):

$$G(s)H(s)=\frac{K e^{-\theta s}}{\tau s+1}.\tag{1}$$

For a single unit heater step $u(t)=\Delta u\,1_{t\ge0}$ starting from ambient temperature $T_a$, the assumed scalar step response is

$$T_{\mathrm{sensor}}(t)=T_a+K\Delta u\left(1-e^{-(t-\theta)_+/\tau}\right),\quad (x)_+=\max(x,0).\tag{2}$$

The parameters reported in **Section 3.1 of the thesis, pages 30–32** are shown below. These are *reported estimates*, not raw training measurements.

In [ ]:
reported = pd.DataFrame([{"case": key, "K": p.K, "tau_s": p.tau, "theta_s": p.delay}
                         for key,p in PLANTS_FROM_THESIS.items()])
print(reported.to_string(index=False))

## 2. Construct an explicitly synthetic four-sensor dataset

TH37 and TH34 use published parameter estimates. TH22 and TH40 parameter choices are **new assumptions** (their individual values were not listed). Gaussian noise and random seed are fixed for teaching purposes. The four-sensor average should **not** be interpreted as matching the paper's separately identified mean-temperature transfer function.

In [ ]:
rng = np.random.default_rng(2021)
t = np.arange(0,3751,5,dtype=float)
assumed = {"TH22": Plant("TH22",6.85,135.,7.5,"synthetic assumption"),
           "TH40": Plant("TH40",7.1852,155.,8.5,"synthetic assumption")}
models = {"TH37": PLANTS_FROM_THESIS["TH37"],"TH34": PLANTS_FROM_THESIS["TH34"],**assumed}
df = pd.DataFrame({"time_s":t})
for key,plant in models.items():
    df[key] = fopdt_step_response(t,plant) + rng.normal(0,.12,len(t))
df["mean_four_sensors"] = df[list(models)].mean(axis=1)
print(df.head(6).to_string(index=False))
fig,ax=plt.subplots(figsize=(10,4))
for name in models: ax.plot(t,df[name],label=name,alpha=.8,lw=1)
ax.set(xlabel="Time (s)",ylabel="Temperature (°C)",title="Synthetic sensor step responses")
ax.grid(alpha=.25);ax.legend();plt.show()

## 3. Identify $K,\tau,\theta$ by least squares

We estimate the model from **synthetic observations only**. Let $\boldsymbol\beta=(K,\tau,\theta)$. Optimize:

$$\hat{\boldsymbol\beta}=\underset{K>0,\tau>0,\theta\ge0}{\arg\min}\sum_i\left[T_i^{\mathrm{synthetic}}-\hat T(t_i;\boldsymbol\beta)\right]^2.\tag{3}$$

This is a **new open implementation**, not a reconstruction of the original MATLAB System Identification Toolbox ARX pipeline.

In [ ]:
from scipy.optimize import least_squares
from burnin_pid_fuzzy.models import Plant
y = df["TH37"].to_numpy()
def residual(z):
    p = Plant("fit",float(z[0]),float(z[1]),float(z[2]))
    return fopdt_step_response(t,p)-y
fit=least_squares(residual,[7.,250.,10.],bounds=([.01,1.,0.],[30.,2000.,100.]))
truth=PLANTS_FROM_THESIS["TH37"]
print("Fitted (synthetic):",dict(zip(["K","tau_s","theta_s"],fit.x.round(4))))
print("Thesis-reported parameters:",truth)
print("RMSE (C):",np.sqrt(np.mean(fit.fun**2)))
fig,ax=plt.subplots(figsize=(10,3.5))
ax.plot(t,y,'.',ms=2,alpha=.45,label="synthetic noisy TH37")
ax.plot(t,fopdt_step_response(t,Plant("fit",*fit.x)),lw=2,label="identified FOPDT")
ax.grid(alpha=.2);ax.legend();ax.set(xlabel="Time (s)",ylabel="Temperature (°C)");plt.show()

## 4. Critical discussion

1. A combined transfer-function fit does not uniquely separate actuator delay from sensor delay.
2. The original identified four cases used different averaging strategies; the mean of four synthetic curves is not automatically equivalent.
3. A step response does not identify spatial thermal gradients. A real deployment needs multi-point validation, heater limits, door instrumentation and safety interlocks.

**Next:** [02 · Mamdani Fuzzy PID](02_fuzzy_pid_and_benchmark.ipynb).